# Visualizing and Interpreting Convolutional Neural Networks

Convolutional Neural Networks (CNNs) are often treated as black boxes that magically map inputs to predictions, but understanding their internal mechanics is essential for building robust models. In this lab, you will peel back the layers to observe exactly how these networks process visual information. You will explore the transformation of images from raw pixels into abstract feature maps and measure how the network's view of the input expands as you go deeper.

You will develop custom tools to intercept the network's internal signals, allowing you to visualize the progression from detecting simple edges to recognizing complex objects. This hands on exploration provides the intuition needed to interpret model behavior and debug complex architectures.

In this lab, you will:

* **Develop Hook Functions**: Create mechanisms to intercept and extract intermediate outputs from specific network layers during the forward pass.
* **Visualize Feature Maps**: Generate visual representations of how the network transforms images step by step to uncover the features detected at various depths.
* **Analyze Receptive Fields**: Compute and interpret the specific region of the input image that influences a given neuron, understanding how local operations build global context.
* **Examine ResNet Internals**: Apply your visualization tools to a pre-trained ResNet model to observe how deep architectures refine data from input to classification.

Let's get started and deepen your understanding of CNNs through visualization!

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as tv_models
from torchvision import transforms

import helper_utils

%matplotlib inline

## Transforming Images Layer by Layer

Your first objective is to observe the transformation pipeline within a CNN. As an image traverses through the network, each layer interprets the data differently. Early layers typically act as detectors for basic structures like edges or textures, while deeper layers aggregate these findings to recognize shapes and objects. You will now visualize these feature maps to see this progressive understanding in action.

In [ ]:
# Getting a ball image to illustrate
ball_image = helper_utils.get_ball()
ball_tensor = torch.tensor(ball_image, dtype=torch.float32)

In [ ]:
helper_utils.plot_image(ball_tensor)

### Visualizing a Convolutional Layer

The convolutional layer is the primary engine of feature extraction. When you pass an image through this layer, it applies a set of learnable filters to the input. Each filter produces a feature map that highlights specific regions where a pattern, such as a vertical line or a color gradient, matches the filter's weights. Visualizing these activation outputs gives you direct insight into what specific characteristics the network is prioritizing at this stage.



In [ ]:
out_channels=16
kernel_size=3 
stride=1
padding=1

# Define a 2D convolutional layer with specified parameters.
conv_layer = nn.Conv2d(
    in_channels=3, 
    out_channels=out_channels, 
    kernel_size=kernel_size, 
    stride=stride, 
    padding=padding
)

# Apply the convolutional layer to the input tensor.
output_conv_layer = conv_layer(ball_tensor)

# Print the shape of the tensor before and after convolution for comparison.
print(f"Input tensor shape: {ball_tensor.shape}")
print(f"Output tensor shape after convolution: {output_conv_layer.shape}")

<br>

As you move deeper into the network, the layers generally possess larger receptive fields and produce smaller feature maps. This trade off allows the model to balance the retention of fine local details with the integration of global context, which is vital for effective pattern interpretation.

In [ ]:
# Loop over each output channel (filter)
for i in range(out_channels):
    # Determine the grid size based on total number of filters
    grid_size = int(np.ceil(np.sqrt(out_channels)))
    # Add a subplot in a grid layout for each filter
    plt.subplot(grid_size, grid_size, i + 1)
    # Detach the tensor from the computation graph, convert to numpy array for visualization
    plt.imshow(output_conv_layer[i].detach().numpy(), cmap='gray')
    # Remove axis for a cleaner look
    plt.axis('off')
    # Set the title for each filter with proper formatting
    plt.title(f'Filter {i+1}', fontsize=10, pad=10)  

# Adjust layout to prevent overlap
plt.tight_layout()  
# Display the plot with all filters
plt.show()
# Adding space
print()

### Downsampling with Pooling Layers

You will now explore the role of pooling layers. These layers are responsible for reducing the spatial dimensions of the feature maps, which lowers computational cost and helps the model achieve translational invariance.

#### Mathematical Definition of Pooling
Pooling functions as a downsampling operation. For a 2D pooling operation with a kernel size $k \times k$ and stride $s$, the output is calculated as follows:

**Max Pooling:**
$$y_{i,j} = \max_{m,n \in R_{i,j}} x_{m,n}$$

**Average Pooling:**
$$y_{i,j} = \frac{1}{k^2} \sum_{m,n \in R_{i,j}} x_{m,n}$$

Here, $R_{i,j}$ represents the pooling region or receptive field for the output position $(i,j)$, defined as:
$$R_{i,j} = \{(m,n) : i \cdot s \leq m < i \cdot s + k, \quad j \cdot s \leq n < j \cdot s + k\}$$

You can calculate the **output dimensions** after pooling using these formulas:
$$H_{out} = \left\lfloor \frac{H_{in} - k}{s} \right\rfloor + 1$$
$$W_{out} = \left\lfloor \frac{W_{in} - k}{s} \right\rfloor + 1$$

In these equations, $H_{in}$ and $W_{in}$ are the input height and width, and $\lfloor \cdot \rfloor$ denotes the floor function.

<br>

#### Example: 2×2 Pooling on a 4×4 Matrix
Consider applying a 2×2 pooling operation with `stride=2` to a 4×4 input matrix.

**Input Matrix (4×4):**
$$X = \left[\begin{array}{cccc}
1 & 3 & 2 & 4 \\
5 & 6 & 7 & 8 \\
9 & 2 & 3 & 1 \\
4 & 5 & 6 & 2
\end{array}\right]$$

**Output Dimensions:**
Using the formulas above:
* $H_{out} = \lfloor \frac{4 - 2}{2} \rfloor + 1 = 1 + 1 = 2$
* $W_{out} = \lfloor \frac{4 - 2}{2} \rfloor + 1 = 1 + 1 = 2$
* The resulting output is a 2×2 matrix.

<br>

**Max Pooling Computations:**

**Position (0,0):** Pool region = rows [0:2], cols [0:2]
$$\left[\begin{array}{cc}
1 & 3 \\
5 & 6
\end{array}\right] \rightarrow \max(1, 3, 5, 6) = 6$$

**Position (0,1):** Pool region = rows [0:2], cols [2:4]
$$\left[\begin{array}{cc}
2 & 4 \\
7 & 8
\end{array}\right] \rightarrow \max(2, 4, 7, 8) = 8$$

**Position (1,0):** Pool region = rows [2:4], cols [0:2]
$$\left[\begin{array}{cc}
9 & 2 \\
4 & 5
\end{array}\right] \rightarrow \max(9, 2, 4, 5) = 9$$

**Position (1,1):** Pool region = rows [2:4], cols [2:4]
$$\left[\begin{array}{cc}
3 & 1 \\
6 & 2
\end{array}\right] \rightarrow \max(3, 1, 6, 2) = 6$$

<br>

**Max Pooling Output (2×2):**
$$Y_{max} = \left[\begin{array}{cc}
6 & 8 \\
9 & 6
\end{array}\right]$$



In [ ]:
# Define the parameters for the max pooling layer
pool_kernel_size = 2  # Size of the pooling window
pool_stride = 2       # Stride of the pooling window

# Create a MaxPool2d layer with specified kernel size and stride
max_pool_layer = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

# Apply the max pooling layer to the input tensor
pooled_tensor = max_pool_layer(ball_tensor)

# Print the tensor shape before and after pooling to observe the dimensionality reduction
print(f"Shape of input tensor before pooling: {ball_tensor.shape}")
print(f"Shape of tensor after pooling: {pooled_tensor.shape}\n")

In [ ]:
# Visualize the original and pooled images side by side
plt.figure(figsize=(12, 6))  # Set figure size for clarity

# Plot original image
plt.subplot(1, 2, 1)
helper_utils.plot_image(ball_tensor, title='Original', aspect='equal')

# Plot pooled image
plt.subplot(1, 2, 2)
helper_utils.plot_image(pooled_tensor, title='Pooled', aspect='equal')

# Adjust layout to prevent overlap and display the figure
plt.tight_layout()
plt.show()

### Building and Understanding a Basic 3-Layer CNN

You will now assemble a simple convolutional neural network with three layers. This foundational architecture illustrates how fundamental operations like convolution, activation functions, and pooling work in concert.

#### Mathematical Definition of 2D Convolution
The 2D convolution is the core operation. Given an input feature map $X$ and a kernel (filter) $K$ of size $k \times k$, the operation is defined as:

$$Y_{i,j} = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} K_{m,n} \cdot X_{i+m, j+n} + b$$

Where:
* $Y_{i,j}$ is the output at position $(i,j)$
* $K_{m,n}$ is the kernel weight at position $(m,n)$
* $X_{i+m, j+n}$ is the input value at position $(i+m, j+n)$
* $b$ is the bias term

**With stride $s$ and padding $p$**, the output dimensions are:
$$H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1$$
$$W_{out} = \left\lfloor \frac{W_{in} + 2p - k}{s} \right\rfloor + 1$$

<br>

#### Example: 3×3 Convolution with 2×2 Kernel
Let's trace a 2×2 kernel applied to a 3×3 input matrix using `stride=1`, `no padding`, and `bias=0`.

**Input Matrix X (3×3):**
$$X = \left[\begin{array}{ccc}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{array}\right]$$

**Kernel K (2×2):**
$$K = \left[\begin{array}{cc}
1 & 0 \\
0 & -1
\end{array}\right]$$

This specific kernel functions as an edge detector by computing the difference between diagonal elements.

**Output Dimensions:**
* $H_{out} = \lfloor \frac{3 + 2(0) - 2}{1} \rfloor + 1 = 1 + 1 = 2$
* $W_{out} = \lfloor \frac{3 + 2(0) - 2}{1} \rfloor + 1 = 1 + 1 = 2$
* The output is a 2×2 matrix.

<br>

**Convolution Computations:**

**Position (0,0):** Apply kernel to top-left 2×2 region
$$\left[\begin{array}{cc}
1 & 2 \\
4 & 5
\end{array}\right] \odot \left[\begin{array}{cc}
1 & 0 \\
0 & -1
\end{array}\right] = (1 \times 1) + (2 \times 0) + (4 \times 0) + (5 \times -1) = 1 - 5 = -4$$

**Position (0,1):** Apply kernel to top-right 2×2 region
$$\left[\begin{array}{cc}
2 & 3 \\
5 & 6
\end{array}\right] \odot \left[\begin{array}{cc}
1 & 0 \\
0 & -1
\end{array}\right] = (2 \times 1) + (3 \times 0) + (5 \times 0) + (6 \times -1) = 2 - 6 = -4$$

**Position (1,0):** Apply kernel to bottom-left 2×2 region
$$\left[\begin{array}{cc}
4 & 5 \\
7 & 8
\end{array}\right] \odot \left[\begin{array}{cc}
1 & 0 \\
0 & -1
\end{array}\right] = (4 \times 1) + (5 \times 0) + (7 \times 0) + (8 \times -1) = 4 - 8 = -4$$

**Position (1,1):** Apply kernel to bottom-right 2×2 region
$$\left[\begin{array}{cc}
5 & 6 \\
8 & 9
\end{array}\right] \odot \left[\begin{array}{cc}
1 & 0 \\
0 & -1
\end{array}\right] = (5 \times 1) + (6 \times 0) + (8 \times 0) + (9 \times -1) = 5 - 9 = -4$$

<br>

**Convolution Output Y (2×2):**
$$Y = \left[\begin{array}{cc}
-4 & -4 \\
-4 & -4
\end{array}\right]$$

The uniform output of -4 suggests that the difference between the top left and bottom right corners is constant across all 2×2 regions in our input. This consistency aligns with the regular consecutive integer pattern in our input matrix.

<br>

#### Example with Different Kernel: Vertical Edge Detection
Now consider a kernel designed to detect vertical edges:

**Kernel K (2×2):**
$$K = \left[\begin{array}{cc}
1 & -1 \\
1 & -1
\end{array}\right]$$

<br>

**Convolution Computations:**

**Position (0,0):**
$$\left[\begin{array}{cc}
1 & 2 \\
4 & 5
\end{array}\right] \odot \left[\begin{array}{cc}
1 & -1 \\
1 & -1
\end{array}\right] = (1 \times 1) + (2 \times -1) + (4 \times 1) + (5 \times -1) = 1 - 2 + 4 - 5 = -2$$

**Position (0,1):**
$$\left[\begin{array}{cc}
2 & 3 \\
5 & 6
\end{array}\right] \odot \left[\begin{array}{cc}
1 & -1 \\
1 & -1
\end{array}\right] = (2 \times 1) + (3 \times -1) + (5 \times 1) + (6 \times -1) = 2 - 3 + 5 - 6 = -2$$

**Position (1,0):**
$$\left[\begin{array}{cc}
4 & 5 \\
7 & 8
\end{array}\right] \odot \left[\begin{array}{cc}
1 & -1 \\
1 & -1
\end{array}\right] = (4 \times 1) + (5 \times -1) + (7 \times 1) + (8 \times -1) = 4 - 5 + 7 - 8 = -2$$

**Position (1,1):**
$$\left[\begin{array}{cc}
5 & 6 \\
8 & 9
\end{array}\right] \odot \left[\begin{array}{cc}
1 & -1 \\
1 & -1
\end{array}\right] = (5 \times 1) + (6 \times -1) + (8 \times 1) + (9 \times -1) = 5 - 6 + 8 - 9 = -2$$

<br>

**Output Y (2×2):**
$$Y = \left[\begin{array}{cc}
-2 & -2 \\
-2 & -2
\end{array}\right]$$

This kernel successfully detects the consistent horizontal gradient present in our input matrix.

**For multiple channels**, if the input has $C_{in}$ channels and you want $C_{out}$ output channels, the formula expands to:
$$Y_{c_{out}, i, j} = \sum_{c_{in}=0}^{C_{in}-1} \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} K_{c_{out}, c_{in}, m, n} \cdot X_{c_{in}, i+m, j+n} + b_{c_{out}}$$

This demonstrates that each output channel is the result of convolving across all input channels with distinct kernels and summing those results.

In [ ]:
class ThreeLayerCNN(nn.Module):
    """
    A three-layer Convolutional Neural Network (CNN).

    This class defines a sequential architecture consisting of three convolutional
    blocks. Each block performs a convolution, applies a non-linear activation, 
    and reduces spatial dimensions via pooling.
    """
    def __init__(self):
        """
        Initialize the ThreeLayerCNN model.
        """
        # Initialize the parent class
        super().__init__()

        # Define the container for the sequence of layers
        self.layers = nn.ModuleList([
            # First convolutional block
            nn.Sequential(
                # Convolutional layer with 3 input channels and 16 output filters
                nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
                # Apply ReLU activation function to introduce non-linearity
                nn.ReLU(),
                # Max pooling to reduce spatial dimensions by a factor of 2
                nn.MaxPool2d(kernel_size=2, stride=2)
            ),
            # Second convolutional block
            nn.Sequential(
                # Convolutional layer taking 16 inputs and producing 32 output filters
                nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
                # Apply ReLU activation function
                nn.ReLU(),
                # Max pooling to reduce spatial dimensions by half
                nn.MaxPool2d(kernel_size=2, stride=2)
            ),
            # Third convolutional block
            nn.Sequential(
                # Convolutional layer taking 32 inputs and producing 64 output filters
                nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
                # Apply ReLU activation function
                nn.ReLU(),
                # Final max pooling layer for size reduction
                nn.MaxPool2d(kernel_size=2, stride=2)
            )
        ])
    
    def forward(self, x):
        """
        Define the forward pass of the network.

        Args:
            x: The input tensor containing image data.

        Returns:
            The output tensor after passing through all network layers.
        """
        # Iterate through each layer in the module list
        for layer in self.layers:
            # Pass the input through the current layer
            x = layer(x)
        # Return the final processed tensor
        return x

### Hook Function

You will now learn to use hook functions in PyTorch, a powerful feature that allows you to inspect the network's internal state. Hooks are functions that you can register to a specific layer to extract, modify, or monitor its outputs during the forward or backward passes. In this context, you will focus on forward hooks, which enable you to capture intermediate outputs at selected layers. This capability is instrumental for visualizing feature maps and understanding the flow of data through the model.

In [ ]:
# Dictionary to store the captured feature maps
activations = {}

def grab(name):
    """
    A function for creating forward hooks to capture intermediate activations.

    This utility generates a closure that allows for the interception of 
    layer outputs during the forward pass, using the provided name as a 
    key for storage.

    Args:
        name: The string identifier used as a key in the activations dictionary.

    Returns:
        A callable hook function suitable for registration on a neural network module.
    """
    def hook(model, input, output):
        """
        Execute the hook logic to process layer outputs.

        Args:
            model: The neural network layer or module triggering the hook.
            input: The input tensor(s) provided to the layer.
            output: The resulting output tensor(s) generated by the layer.
        """
        # Detach the output tensor from the computation graph and save it to the global dictionary
        activations[name] = output.detach()

    # Return the inner hook function to be registered
    return hook

In [ ]:
model = ThreeLayerCNN()

#### Registering Forward Hooks

In this step, you will register forward hooks on the specific layers you wish to examine. Forward hooks capture the layer's output during the forward pass, providing the data needed to visualize feature maps or analyze intermediate activations.

To achieve this, you call the `register_forward_hook()` method on the desired layer, passing in a hook function that executes whenever that layer performs a forward pass. This function receives the layer, its input, and its output, allowing you to save the output for later analysis.

Ensure you register these hooks **before** passing data through the model to guarantee you capture the intermediate representations.

In [ ]:
# Register forward hooks on specific layers to capture activations during a forward pass
# This helps us visualize what features each layer detects
model.layers[0].register_forward_hook(grab('layer1'))  # Hook for layer1
model.layers[1].register_forward_hook(grab('layer2'))  # Hook for layer2
model.layers[2].register_forward_hook(grab('layer3'))  # Hook for layer3

In [ ]:
# Passing the ball tensor to the network
output = model(ball_tensor.unsqueeze(0))
print(output.shape)

In [ ]:
# Printing the shapes after each layer
for layer, output_tensor in activations.items():
    print(f"Output shape after {layer}: {output_tensor.shape}")

In [ ]:
helper_utils.visualize_all_layers_grids(activations)

## Receptive Field in Convolutional Neural Networks

### What is the *Receptive Field*?

The **receptive field** is the specific region of the input image that influences a particular neuron in a feature map. It represents the "context" or area of the input that a unit at a specific layer can see.

* **Small receptive field**: The unit sees only a tiny fraction of the input.
* **Large receptive field**: The unit captures a broader area, integrating global image information.

### Why is the receptive field important?

It reveals **how much of the input image** contributes to a specific feature at a given layer. Larger receptive fields enable the network to recognize complex, global patterns. If the field is too small, the network might fail to capture vital global context.

### How is the receptive field computed?

You can calculate the receptive field recursively as layers are stacked. While a single convolutional layer increases the field based on its kernel size, subsequent layers—especially those following pooling or strided convolutions—see an exponentially growing region of the original input.

Think of it this way: a filter in a deeper layer looks at a feature map where each "pixel" already represents a pooled region of the original image. Therefore, a small 3x3 kernel in a deep layer effectively spans a much larger patch of the original pixels.

#### **Key Terms:**
* **Kernel:** The filter window (e.g., 3×3) extracting features.
* **Stride:** The step size of the kernel movement.
* **Dilation:** The spacing between kernel elements.

#### **Recursive Calculation Rule:**

Given a layer with kernel size $ k $, stride $ s $, dilation $ d $, previous receptive field $ r_{prev} $, and previous jump $ j_{prev} $:

$$
r_{new} = r_{prev} + (k - 1) \times j_{prev} \times d
$$
$$
j_{new} = j_{prev} \times s
$$

* **Receptive field ($r$)** grows based on the kernel size and the cumulative stride.
* **Jump ($j$)** is the effective stride between neighboring units in the current feature map relative to the input.

#### **Example Calculation**

Consider a network processing a 224x224 image:

| Layer type | Kernel | Stride | Padding |
| :--- | :--- | :--- | :--- |
| Input | N/A | N/A | N/A |
| Conv2d | 3 | 1 | 1 |
| MaxPool2d | 2 | 2 | 0 |

* Start with $ r = 1 $, $ j = 1 $ (each pixel sees itself).
* **After conv layer** ($ k=3, s=1 $):
    $ r = 1 + (3-1)*1 = 3 $, $ j = 1*1 = 1 $
* **After pool layer** ($ k=2, s=2 $):
    $ r = 3 + (2-1)*1 = 4 $, $ j = 1*2 = 2 $

After these two layers, each neuron in the feature map sees a 4x4 patch of the original input, with neighboring neurons spaced 2 pixels apart in the input space.

### **Summary:**

* The **receptive field** quantifies the input region contributing to a feature.
* It **grows gradually** with depth, driven by kernel size, pooling, and strides.
* Mastering this concept allows you to design networks that capture the appropriate amount of context for your task.

Let's visualize this growth!



In [ ]:
rfinfo = helper_utils.calculate_receptive_field(model, input_size=224)
helper_utils.plot_receptive_field_summary(rfinfo)

## A Practical Example with ResNet

You will now investigate how ResNet processes images internally. You'll trace the transformation of an image through the network's layers and analyze the Receptive Field graph to determine the context available to each neuron. This practical example elucidates ResNet's architecture and the flow of information from input to final prediction.

In [ ]:
# Load pre-trained ResNet50
torch.hub.set_dir('pretrained_model')
model = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2).eval()

In [ ]:
help(tv_models.resnet50)

### Plotting the Receptive Field Map

Here you will visualize the receptive field of neurons across different network layers. This visualization demonstrates the spatial extent of the input image that each neuron can "see," helping you understand the scale of features captured at various depths.

In [ ]:
rfinfo = helper_utils.calculate_receptive_field(model, input_size=224)
helper_utils.plot_receptive_field_summary(rfinfo)

### Analyzing Selected Layers of ResNet

You will now examine the progressive transformation of an image as it traverses the ResNet model. You will focus on how features evolve from the raw input to the final prediction. Since ResNet contains many layers, you will select a few representative ones for this detailed analysis.

#### Applying Forward Hooks to ResNet Layers

You will apply forward hooks to specific layers within the ResNet architecture to capture their outputs.

In [ ]:
# Reset activations dictionary
activations = {}
# To register a forward hook, you need to call the following method on each layer you want to register
model.conv1.register_forward_hook(grab('conv1'))           # First layer
model.layer1[0].conv1.register_forward_hook(grab('layer1'))  # Early block
model.layer2[0].conv1.register_forward_hook(grab('layer2'))  # Middle block
model.layer3[0].conv1.register_forward_hook(grab('layer3'))  # Later block
model.layer4[0].conv1.register_forward_hook(grab('layer4'))  # Deep block

#### Preprocessing Pipeline

Before passing images to ResNet, you must preprocess them to align with the model's training conditions. This typically entails resizing, cropping, normalization, and conversion to tensors. The standard steps are:

1.  **Resize**: Scale the image so the shorter side is 256 pixels.
2.  **Center Crop**: Extract a 224x224 pixel region from the center.
3.  **To Tensor**: Convert the PIL image into a PyTorch tensor.
4.  **Normalize**: Scale pixel values using the ImageNet mean and standard deviation (mean: [0.485, 0.456, 0.406], std: [0.229, 0.224, 0.225]).

These steps ensure the input data is properly scaled and normalized, allowing the model to perform accurately.

In [ ]:
# Load and preprocess an image
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load a sample image (let's say we have a cat image)
image = Image.open('images/cat.jpg')
input_tensor = transform(image).unsqueeze(0)  # Add batch dimension

In [ ]:
# Forward pass through the model
with torch.no_grad():
    output = model(input_tensor)

# Visualize the input image and feature maps
plt.figure(figsize=(15, 10))

# Display original image
plt.subplot(2, 3, 1)
plt.imshow(image)
plt.title('Original Image')
plt.axis('off')

### Exploring Feature Maps and Receptive Fields in ResNet

You will now peek inside ResNet to view its feature maps and analyze neuron responses across different layers. Examining these maps clarifies how ResNet deciphers complex images and identifies the types of features detected at each level. By correlating feature maps with receptive fields, you can see how ResNet constructs a detailed understanding of visual data step by step.

In [ ]:
helper_utils.visualize_all_layers_grids(activations)

In [ ]:
helper_utils.plot_rfinfo_over_image(rfinfo, 'images/cat.jpg', input_size=224)

### Interactive Visualization of Image Transformations in ResNet

This widget allows you to track the transformation of an input image at each layer of the ResNet model. You can step through the layers to witness feature extraction and the refinement of representations, leading up to the final classification.

In [ ]:
helper_utils.plot_widget(model)

## Conclusion

In this lab, you have peeled back the layers of a Convolutional Neural Network to expose its internal mechanisms. You observed how early layers act as detectors for simple features like edges and textures, while deeper layers aggregate these into complex object recognizers.

By calculating and visualizing the receptive field, you gained insight into the hierarchical nature of CNNs, seeing how the "view" of each neuron expands from a few pixels to the entire image. Finally, by applying these tools to a pre-trained ResNet, you witnessed how state of the art models build a comprehensive understanding of the visual world, layer by layer.

These visualization techniques are powerful assets for debugging, interpreting, and refining neural networks, helping you build models that are not only accurate but also explainable.